# 03 — Test-set evaluation

Runs sliding-window inference on the held-out test set for a trained checkpoint,
applies the F1-optimised and recall-optimised operating points selected by `02_validation_sweep.ipynb`,
and produces the thesis Test-set tables (4.3, 4.13) and AP-vs-tolerance curve (Fig 4.10).

**Usage**: set `CHECKPOINT_PATH`, `EXPERIMENT`, and the two operating points, then run top-down.


## 1. Config

In [ ]:
# ============================ EDIT THESE =====================================
CHECKPOINT_PATH = "/home/jinny/aspotting/checkpoints/exp3_r2plus1d_1shot/<TIMESTAMP>/best.pt"
EXPERIMENT      = "exp3_r2plus1d_1shot"
BACKBONE        = "r2plus1d_18"
TASK            = "4class"          # "binary" | "4class"

# Operating points selected on the validation sweep (see 02_validation_sweep.ipynb top-5 tables).
# Example values come from Table 4.12 in the thesis.
OPERATING_POINTS = {
    "F1-opt": {"thresh": 0.22, "nms_s": 45.0, "smooth_k": 7},
    "R-opt":  {"thresh": 0.20, "nms_s": 25.0, "smooth_k": 1},
}
# =============================================================================

DATA_DIR    = "/home/jinny/aspotting/dataset"   # SoccerNet root with league/season/game/
OUTPUT_DIR  = f"/home/jinny/aspotting/results/test_predictions/{EXPERIMENT}"

# Must match training (Table 3.3)
FPS, CLIP_SEC, CLIP_FRAMES, CLIP_SIZE = 25, 4.0, 16, (112, 112)
STRIDE_S    = 0.5
BATCH_SIZE  = 64

# Tolerance windows for Tight / Loose Avg-AP (§3.5.4)
TIGHT_TOLS = [1, 2, 3, 4, 5]
LOOSE_TOLS = list(range(5, 61, 5))
TOL_FIXED  = 5                  # for P, R, F1, FP/half


## 2. Imports + load model

In [ ]:
import os, json, pickle
from collections import defaultdict, deque
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision.models.video import (
    r2plus1d_18, R2Plus1D_18_Weights,
    r3d_18,      R3D_18_Weights,
    mc3_18,      MC3_18_Weights,
)
from tqdm.auto import tqdm

DATA_DIR = Path(DATA_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MEAN   = np.array([0.43216, 0.394666, 0.37645],  dtype=np.float32)
STD    = np.array([0.22803, 0.22145,  0.216989], dtype=np.float32)

BACKBONES = {"r2plus1d_18": (r2plus1d_18, R2Plus1D_18_Weights.KINETICS400_V1),
             "r3d_18":      (r3d_18,      R3D_18_Weights.KINETICS400_V1),
             "mc3_18":      (mc3_18,      MC3_18_Weights.KINETICS400_V1)}

factory, weights = BACKBONES[BACKBONE]
model = factory(weights=weights)

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
in_features = model.fc.in_features
if "fc.weight" in state:
    OUT_FEATURES = int(state["fc.weight"].shape[0])
    model.fc = nn.Linear(in_features, OUT_FEATURES)
else:
    OUT_FEATURES = int(state["fc.1.weight"].shape[0])
    model.fc = nn.Sequential(nn.Dropout(p=0.4), nn.Linear(in_features, OUT_FEATURES))
model.load_state_dict(state, strict=True)
model.to(DEVICE).eval()

USE_SIGMOID = (OUT_FEATURES == 1)
print(f"Loaded   : {CHECKPOINT_PATH}")
print(f"Head     : {'sigmoid 1-out' if USE_SIGMOID else f'softmax {OUT_FEATURES}-out'}")


## 2b. Download test split if missing

In [ ]:
DOWNLOAD_IF_MISSING = True   # set False to skip; SoccerNet downloader skips existing files anyway
SPLITS_TO_FETCH     = ['test']

if DOWNLOAD_IF_MISSING:
    try:
        from SoccerNet.Downloader import SoccerNetDownloader
    except ImportError:
        raise ImportError("pip install SoccerNet")
    nv_pw = os.environ.get("NV_PASSWORD")
    if not nv_pw:
        raise RuntimeError(
            "Set NV_PASSWORD in env (register at https://www.soccer-net.org/data). "
            "Example: export NV_PASSWORD=...   or set DOWNLOAD_IF_MISSING=False to skip."
        )
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    d = SoccerNetDownloader(LocalDirectory=str(DATA_DIR))
    d.password = nv_pw
    print(f"Downloading SoccerNet splits {SPLITS_TO_FETCH} into {DATA_DIR} …")
    d.downloadGames(files=["Labels-v2.json"],                  split=SPLITS_TO_FETCH)
    d.downloadGames(files=["1_224p.mkv", "2_224p.mkv"],         split=SPLITS_TO_FETCH)
    print("Download step done (existing files were skipped).")


## 3. Helpers (same as in 02_)

In [ ]:
SHOT_ON_LABELS  = {"shots on target", "penalty"}
SHOT_OFF_LABELS = {"shots off target"}


def parse_annotations(labels_path):
    with open(labels_path) as f:
        data = json.load(f)
    events = defaultdict(list)
    for ann in data["annotations"]:
        label = ann.get("label", "").strip().lower()
        half_str, _ = ann["gameTime"].split(" - ")
        half  = int(half_str)
        pos_s = int(ann["position"]) / 1000.0
        if TASK == "binary":
            if label == "goal":
                events[half].append((pos_s, 1))
        else:
            if label == "goal":
                events[half].append((pos_s, 3))
            elif label in SHOT_ON_LABELS:
                events[half].append((pos_s, 2))
            elif label in SHOT_OFF_LABELS:
                events[half].append((pos_s, 1))
    return dict(events)


def _moving_average(arr, k):
    if k <= 1: return arr
    pad = k // 2
    padded = np.pad(arr, (pad, pad), mode="edge")
    return np.convolve(padded, np.ones(k, dtype=np.float32) / k, mode="valid")


def sliding_window_inference(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []
    fps_v         = cap.get(cv2.CAP_PROP_FPS) or FPS
    total_frames  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step          = max(1, int(CLIP_SEC * fps_v / CLIP_FRAMES))
    window_frames = step * CLIP_FRAMES
    stride_frames = max(1, int(STRIDE_S * fps_v))

    frame_buf, raw = deque(maxlen=window_frames), []
    batch_clips, batch_centers = [], []

    def _flush():
        if not batch_clips: return
        t = torch.stack(batch_clips).float().to(DEVICE)
        with torch.no_grad(), torch.amp.autocast(DEVICE):
            logits = model(t)
            if USE_SIGMOID:
                gp = torch.sigmoid(logits[:, 0]).cpu().numpy()
                probs = np.stack([1 - gp, gp], axis=1)
            else:
                probs = torch.softmax(logits, dim=1).cpu().numpy()
        for cs, class_probs in zip(batch_centers, probs):
            raw.append((float(cs), class_probs.tolist()))
        batch_clips.clear(); batch_centers.clear()

    first_emit = window_frames - 1
    for frame_idx in tqdm(range(total_frames), desc=Path(video_path).name,
                          leave=False, unit="fr"):
        ok, frame = cap.read()
        if not ok: break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, CLIP_SIZE)
        frame_buf.append(frame)
        if frame_idx >= first_emit and (frame_idx - first_emit) % stride_frames == 0:
            buf = list(frame_buf)
            frames = [buf[i * step] for i in range(CLIP_FRAMES)]
            arr = (np.stack(frames).astype(np.float32) / 255.0 - MEAN) / STD
            clip = torch.from_numpy(arr).permute(3, 0, 1, 2)
            cs = (frame_idx - window_frames // 2) / fps_v
            batch_clips.append(clip); batch_centers.append(cs)
            if len(batch_clips) == BATCH_SIZE:
                _flush()
    cap.release(); _flush()
    return sorted(raw, key=lambda x: x[0])


def postprocess_goal(raw_curve, thresh, nms_s, smooth_k):
    if not raw_curve: return []
    goal_cid = 1 if TASK == "binary" else 3
    cs_arr = [x[0] for x in raw_curve]
    p_arr  = np.array([x[1][goal_cid] for x in raw_curve], np.float32)
    if smooth_k > 1:
        p_arr = _moving_average(p_arr, smooth_k)
    candidates = sorted(zip(cs_arr, p_arr.tolist()), key=lambda x: -x[1])
    dets = []
    for cs, prob in candidates:
        if prob < thresh: continue
        if all(abs(cs - d["timestamp_s"]) >= nms_s for d in dets):
            dets.append({"timestamp_s": cs, "confidence": float(prob)})
    dets.sort(key=lambda x: x["timestamp_s"])
    return dets


try:
    _trapz = np.trapezoid
except AttributeError:
    _trapz = np.trapz


def compute_ap_at_tol(preds_with_conf, gt_timestamps, tol):
    if not gt_timestamps:
        return float("nan")
    preds = sorted(preds_with_conf, key=lambda x: -x[1])
    matched, tp_list, fp_list = set(), [], []
    for pred_t, _ in preds:
        best, best_d = None, tol
        for i, gt_t in enumerate(gt_timestamps):
            d = abs(pred_t - gt_t)
            if i not in matched and d <= best_d:
                best_d, best = d, i
        if best is not None:
            matched.add(best); tp_list.append(1); fp_list.append(0)
        else:
            tp_list.append(0); fp_list.append(1)
    tp_cum, fp_cum = np.cumsum(tp_list), np.cumsum(fp_list)
    prec = tp_cum / (tp_cum + fp_cum)
    rec  = tp_cum / len(gt_timestamps)
    prec = np.concatenate([[1.0], prec])
    rec  = np.concatenate([[0.0], rec])
    return float(_trapz(prec, rec))


## 4. Discover test games

In [ ]:
try:
    from SoccerNet.Downloader import getListGames
except ImportError:
    raise ImportError("pip install SoccerNet")

testset_root = DATA_DIR
official = getListGames("test")
games = [testset_root / g for g in official
         if (testset_root / g / "Labels-v2.json").exists()
         and any((testset_root / g).glob("*_224p.mkv"))]
print(f"Found {len(games)} / {len(official)} official test games at {DATA_DIR}")


## 5. Sliding-window inference on test set → pickle (slow; do once)

In [ ]:
RAW_PICKLE = Path(OUTPUT_DIR) / "raw_curves_test.pkl"
FORCE_RERUN = False

if RAW_PICKLE.exists() and not FORCE_RERUN:
    with open(RAW_PICKLE, "rb") as f:
        saved = pickle.load(f)
    all_raw, all_gt = saved["all_raw"], saved["all_gt"]
    print(f"Loaded cached raw curves from {RAW_PICKLE}  ({len(all_raw)} halves)")
else:
    all_raw, all_gt = {}, {}
    goal_cid = 1 if TASK == "binary" else 3
    for game_dir in tqdm(games, desc="games"):
        annotations = parse_annotations(game_dir / "Labels-v2.json")
        for half in (1, 2):
            vid = game_dir / f"{half}_224p.mkv"
            if not vid.exists():
                continue
            key = (str(game_dir.relative_to(testset_root)), half)
            all_raw[key] = sliding_window_inference(str(vid))
            half_evs    = annotations.get(half, [])
            all_gt[key] = [t for t, c in half_evs if c == goal_cid]
    with open(RAW_PICKLE, "wb") as f:
        pickle.dump({"all_raw": all_raw, "all_gt": all_gt,
                     "checkpoint": CHECKPOINT_PATH, "task": TASK}, f)
    print(f"\nSaved: {RAW_PICKLE}  ({len(all_raw)} halves)")


## 6. Apply operating points → headline test-set table (4.3 / 4.13)

In [ ]:
def evaluate_op_point(all_raw, all_gt, thresh, nms_s, smooth_k):
    preds_all, gt_all = [], []
    tp = fp = fn = 0
    n_halves = len(all_raw)
    for key, raw_curve in all_raw.items():
        dets = postprocess_goal(raw_curve, thresh, nms_s, smooth_k)
        gt   = all_gt.get(key, [])
        preds_all.extend((d["timestamp_s"], d["confidence"]) for d in dets)
        gt_all.extend(gt)
        missed = [g for g in gt
                  if not any(abs(g - d["timestamp_s"]) <= TOL_FIXED for d in dets)]
        tp += len(gt) - len(missed)
        fp += sum(1 for d in dets
                  if not any(abs(d["timestamp_s"] - g) <= TOL_FIXED for g in gt))
        fn += len(missed)
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    tight = float(np.nanmean([compute_ap_at_tol(preds_all, gt_all, t) for t in TIGHT_TOLS]))
    loose = float(np.nanmean([compute_ap_at_tol(preds_all, gt_all, t) for t in LOOSE_TOLS]))
    return dict(P=round(p, 4), R=round(r, 4), F1=round(f1, 4),
                Tight_AvgAP=round(tight, 4), Loose_AvgAP=round(loose, 4),
                FP_per_half=round(fp / max(n_halves, 1), 2),
                TP=tp, FP=fp, FN=fn)


rows = []
for name, op in OPERATING_POINTS.items():
    metrics = evaluate_op_point(all_raw, all_gt, op["thresh"], op["nms_s"], op["smooth_k"])
    rows.append({"experiment": EXPERIMENT, "config": name, **op, **metrics})

df = pd.DataFrame(rows)
out_csv = Path(OUTPUT_DIR) / "headline_metrics.csv"
df.to_csv(out_csv, index=False)

print(df.to_string(index=False))
print(f"\nSaved: {out_csv}")


## 7. AP-vs-tolerance curve (Fig 4.10) — produced from raw pickle, no retraining

In [ ]:
import matplotlib.pyplot as plt

op = OPERATING_POINTS["F1-opt"]
preds_all, gt_all = [], []
for key, raw_curve in all_raw.items():
    dets = postprocess_goal(raw_curve, op["thresh"], op["nms_s"], op["smooth_k"])
    preds_all.extend((d["timestamp_s"], d["confidence"]) for d in dets)
    gt_all.extend(all_gt.get(key, []))

ALL_TOLS = sorted(set(TIGHT_TOLS + LOOSE_TOLS))
ap_curve = {t: compute_ap_at_tol(preds_all, gt_all, t) for t in ALL_TOLS}

curve_df = pd.DataFrame({"tolerance_s": ALL_TOLS,
                         "AP": [ap_curve[t] for t in ALL_TOLS]})
curve_csv = Path(OUTPUT_DIR) / "ap_vs_tolerance.csv"
curve_df.to_csv(curve_csv, index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ALL_TOLS, [ap_curve[t] for t in ALL_TOLS], marker="o", linewidth=1.5)
ax.axvspan(TIGHT_TOLS[0], TIGHT_TOLS[-1], alpha=0.15, label="tight (1-5 s)")
ax.axvline(TIGHT_TOLS[-1], color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("Temporal tolerance (s)")
ax.set_ylabel("Goal Average Precision")
ax.set_title(f"{EXPERIMENT}  |  F1-opt config")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig_path = Path(OUTPUT_DIR) / "ap_vs_tolerance.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print(f"\nSaved curve CSV : {curve_csv}")
print(f"Saved figure    : {fig_path}")
